# GitHub Contributor Nonlinear Performance Analysis

Fetches contributor statistics from a GitHub repository and fits the nonlinear maintenance model

$$w(t) = \frac{\text{amplitude}}{\mu}\left(1 - e^{-\mu t}\right)$$

to each contributor's cumulative activity over time.

## Configuration

Set `REPO` to `"owner/repo"`. Optionally set `GITHUB_TOKEN` to avoid rate limiting (required for private repos).

In [ ]:
REPO = "torvalds/linux"  # change to any public repo, e.g. "pandas-dev/pandas"
GITHUB_TOKEN = ""        # optional: set a personal access token to raise rate limits

In [ ]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

## Fetch contributor statistics

GitHub may return HTTP 202 while it computes statistics for the first time — the cell retries until data is ready.

In [ ]:
def fetch_contributors(repo, token="", max_retries=6, retry_delay=5):
    """Fetch contributor weekly stats from the GitHub API.

    Returns a DataFrame where each row is a contributor with a 'weeks' column
    containing a list of {w, a, d, c} dicts (matching linux_contributors.json format).
    """
    url = f"https://api.github.com/repos/{repo}/stats/contributors"
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    for attempt in range(1, max_retries + 1):
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            data = response.json()
            if data:  # empty list means no contributors
                return pd.DataFrame(data)
            raise ValueError(f"Repository '{repo}' has no contributor data.")
        if response.status_code == 202:
            print(f"GitHub is computing stats, retrying in {retry_delay}s (attempt {attempt}/{max_retries})...")
            time.sleep(retry_delay)
            continue
        if response.status_code == 404:
            raise ValueError(f"Repository '{repo}' not found. Check the owner/repo spelling.")
        if response.status_code == 403:
            raise PermissionError("Rate limit exceeded or access denied. Set GITHUB_TOKEN to a valid token.")
        response.raise_for_status()

    raise TimeoutError(f"GitHub did not return contributor stats after {max_retries} attempts.")


contributors = fetch_contributors(REPO, token=GITHUB_TOKEN)
print(f"Fetched {len(contributors)} contributors from '{REPO}'")

## Model

In [ ]:
def fun(t, amplitude, mu):
    # t is in weeks; divide by 52 to convert to years (mu is a per-year rate)
    return amplitude / mu * (1 - np.exp(-mu * t / 52))


def lsq(x_observed, params):
    t = np.arange(len(x_observed))
    x_model = fun(t, *params)
    err = x_model - x_observed
    return np.dot(err, err) / len(err)


def model_fit(x, title=""):
    x = np.array(x, dtype=np.float64)
    x_max = x.max()
    if x_max == 0:
        return None
    x = x / x_max
    res = minimize(
        lambda params: lsq(x, params),
        (1 / len(x), 0.1),
        method='SLSQP',
        bounds=((0, None), (0, None))
    )
    amplitude, mu = res.x
    # effective maintenance ratio: mu normalized by amplitude scale factor
    mu_effective = mu / amplitude
    mx = fun(np.arange(len(x)), amplitude, mu)
    label = f"{title} — " if title else ""
    pd.Series(mx).plot(title=r"{}$\mu$ = {:3.2f}".format(label, mu_effective))
    pd.Series(x).plot()
    plt.show()
    return res

## Run analysis

In [ ]:
def main(df):
    for _, entry in df.iterrows():
        rm = dict()
        for week in entry.weeks:
            d = dict(week)
            w = d['w']
            if w in rm:
                rm[w] += d['a'] + d['d']
            else:
                rm[w] = d['a'] + d['d']
        sorted_items = sorted(rm.items())
        t = pd.Series([v for _, v in sorted_items]).cumsum()
        t = t[t != 0]  # Remove zero-activity weeks
        login = entry.get('author', {}) or {}
        if isinstance(login, dict):
            login = login.get('login', '')
        model_fit(t, title=login)


plt.rcParams["figure.figsize"] = (9, 4)
plt.style.use('fivethirtyeight')
main(contributors)